# 개별종목 조합D — XGBoost

`기본모델/03.XGBoost.ipynb`과 같은 `models.xgboost.build_xgboost_baseline`을 가져오고
조합D 피처를 주입합니다. 기본모델 코드는 `models/`에 한 번만 존재합니다.
후보·라벨·날짜 그룹 12폴드 실행은 모든 조합이 같은 공통 함수를 사용합니다.


In [1]:
# 1. 기본모델을 가져옵니다.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.xgboost import build_xgboost_baseline  # noqa: E402

MODEL_NAME = 'XGBoost'
MODEL_BUILDER = build_xgboost_baseline


In [2]:
# 2. 조합D의 피처 값만 지정합니다.
import json

COMBINATION = 'D'
FEATURE_COLUMNS = (
    'atr_ratio',
    'hv_20',
    'range_1',
    'range_20',
    'bb_bandwidth',
    'volume_z_20',
    'turnover_20',
    'log_amihud_20',
)

report_path = project_root / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
print(f"조합{COMBINATION} 피처:", FEATURE_COLUMNS)
combination_report = report["combinations"].get(COMBINATION)
if combination_report is None:
    print("아직 실측 결과가 없습니다. 아래 공통 실행 명령으로 조합을 평가하세요.")
else:
    panel = combination_report["panel"]
    print("학습 기간:", panel["first_date"], "~", panel["last_date"])
    print("학습 행·종목:", panel["model_rows"], panel["stocks"])
    folds = pd.DataFrame(combination_report["outer_fold_results"])
    model_folds = folds.loc[folds["model"].eq(MODEL_NAME)].reset_index(drop=True)
    fold_columns = [
        "fold", "selected_class_weight", "train_dates", "valid_start", "valid_end",
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, fold_columns].round(4))
    metric_columns = [
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, metric_columns].mean().to_frame("OOS 폴드 평균").round(4))

# 조합별 노트북이 중복 학습하지 않도록 실제 fit은 공통 실행기에서 한 번 수행합니다.
print("재실행 명령: python scripts/run_stock_model_experiment.py")


조합D 피처: ('atr_ratio', 'hv_20', 'range_1', 'range_20', 'bb_bandwidth', 'volume_z_20', 'turnover_20', 'log_amihud_20')
학습 기간: 20110127 ~ 20240822
학습 행·종목: 159900 157


,fold,selected_class_weight,train_dates,valid_start,valid_end,accuracy,training_majority_baseline_accuracy,accuracy_minus_training_majority_baseline,macro_f1,balanced_accuracy,mcc,pr_auc_macro_ovr,down_recall,core_harmonic_mean
0,1,NaN,750,20140217,20140514,0.5064,0.5012,0.0052,0.2838,0.3540,0.0798,0.3801,0.0571,0.1304
1,2,balanced,980,20150123,20150421,0.3996,0.3978,0.0018,0.3515,0.3663,0.0596,0.3787,0.1307,0.2308
2,3,NaN,1210,20151228,20160328,0.3840,0.3762,0.0079,0.3312,0.3610,0.0508,0.3766,0.1755,0.2650
3,4,balanced,1439,20161202,20170228,0.4537,0.4617,-0.0081,0.3509,0.3738,0.0791,0.4171,0.1207,0.2249
4,5,balanced,1669,20171113,20180207,0.4033,0.3901,0.0132,0.3705,0.3832,0.0837,0.3916,0.2079,0.3003
5,6,NaN,1899,20181024,20190118,0.4067,0.3725,0.0342,0.3789,0.3909,0.0962,0.4134,0.2420,0.3250
6,7,balanced,2129,20190930,20191224,0.4607,0.4781,-0.0174,0.3450,0.3670,0.0801,0.4009,0.1185,0.2221
7,8,NaN,2359,20200902,20201130,0.3731,0.3476,0.0255,0.3634,0.3737,0.0629,0.3820,0.3342,0.3561
8,9,NaN,2589,20210806,20211105,0.4137,0.3914,0.0223,0.3599,0.3847,0.0925,0.3923,0.2128,0.3032
9,10,NaN,2818,20220714,20221012,0.3651,0.3454,0.0197,0.3298,0.3585,0.0431,0.3755,0.2099,0.2848


,OOS 폴드 평균
accuracy,0.4116
training_majority_baseline_accuracy,0.3969
accuracy_minus_training_majority_baseline,0.0147
macro_f1,0.3513
balanced_accuracy,0.3734
mcc,0.0742
pr_auc_macro_ovr,0.3915
down_recall,0.1993
core_harmonic_mean,0.2776


재실행 명령: python scripts/run_stock_model_experiment.py
